# Phase 1: Data Infrastructure & Empirical Diagnostics  
## Notebook 01_05 — Empirical Distribution Analyzer: Normal vs Student-t vs Laplace

### Research question

How well do simple parametric distributions describe cleaned financial returns, and what goes wrong when we assume returns are normally distributed?

This notebook compares three return-distribution models:

1. **Normal distribution** — thin-tailed benchmark;
2. **Student-t distribution** — heavy-tailed symmetric model;
3. **Laplace distribution** — sharper peak with exponential tails.

The notebook builds a reproducible empirical distribution analyzer that:

- generates or loads a cleaned return series;
- computes descriptive statistics;
- fits Normal, Student-t, and Laplace distributions;
- compares log-likelihood, AIC, BIC, and out-of-sample likelihood;
- visualises PDFs, QQ plots, and probability integral transforms;
- compares empirical and model-implied tail probabilities;
- estimates Value-at-Risk and Expected Shortfall;
- saves a model-comparison report for later risk notebooks.

The goal is not to crown a single universal best distribution. The goal is to show that distributional assumptions are model-risk assumptions.

## 1. Why distribution analysis matters

Many finance models start by assuming returns are normally distributed.

That assumption is mathematically convenient, but financial returns often display:

- heavy tails;
- excess kurtosis;
- volatility clustering;
- jumps;
- skewness;
- regime changes;
- crisis periods.

If the model underestimates tail probability, then risk systems can underestimate losses.

This matters for:

- Value-at-Risk;
- Expected Shortfall;
- option pricing;
- stress testing;
- portfolio construction;
- backtest realism;
- margin and leverage decisions.

A useful habit is:

> Before modelling alpha, inspect the empirical distribution of the returns the strategy will actually consume.

## 2. The three candidate distributions

### 2.1 Normal distribution

The Normal model assumes:

$$
r \sim \mathcal N(\mu, \sigma^2)
$$

with density:

$$
\begin{aligned}
f(r) &= \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{1}{2}\left(\frac{r-\mu}{\sigma}\right)^2\right)
\end{aligned}
$$

It is symmetric and thin-tailed.

### 2.2 Student-t distribution

The Student-t model can be written as:

$$
r \sim t_\nu(\mu, s)
$$

where:

- $\nu$ is the degrees-of-freedom parameter;
- $\mu$ is location;
- $s$ is scale.

Lower $\nu$ means heavier tails. As:

$$
\nu \to \infty
$$

the Student-t distribution approaches the Normal distribution.

### 2.3 Laplace distribution

The Laplace distribution has density:

$$
\begin{aligned}
f(r) &= \frac{1}{2b} \exp\left(-\frac{|r-\mu|}{b}\right)
\end{aligned}
$$

It is symmetric, more sharply peaked than the Normal distribution, and has heavier tails than the Normal but usually lighter tails than low-degree-of-freedom Student-t models.

### 2.4 Model risk warning

All three distributions are simplified.

They are unconditional, symmetric, and single-regime models. Real returns may be conditional on volatility state, market regime, liquidity, and macro conditions.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy import stats
except ImportError as exc:
    raise ImportError(
        "This notebook requires scipy. Install with: pip install scipy"
    ) from exc

rng = np.random.default_rng(seed=42)

## 3. Load or generate returns

This notebook can work in two modes:

1. **Load cleaned returns** from the previous sanitization notebook if available.
2. **Generate synthetic heavy-tailed returns** if no previous data file exists.

The fallback synthetic data is useful because this notebook should be runnable on its own.

In [ ]:
def load_previous_sanitized_returns(path: Path) -> pd.DataFrame | None:
    # Load sanitized returns from Notebook 01_04 if the file exists.
    if not path.exists():
        return None

    data = pd.read_csv(path)

    required_cols = {"symbol", "timestamp", "sanitized_log_return", "return_is_usable"}
    missing = required_cols - set(data.columns)

    if missing:
        warnings.warn(f"Previous sanitized file is missing columns: {missing}")
        return None

    data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True)
    data = data[data["return_is_usable"] == True].copy()
    data = data.replace([np.inf, -np.inf], np.nan).dropna(subset=["sanitized_log_return"])

    return data[["symbol", "timestamp", "sanitized_log_return"]].rename(
        columns={"sanitized_log_return": "log_return"}
    )

In [ ]:
def simulate_heavy_tailed_return_panel(
    num_days: int = 1_500,
    num_assets: int = 60,
    seed: int = 42
) -> pd.DataFrame:
    # Generate a synthetic panel of returns with volatility regimes and occasional jumps.
    local_rng = np.random.default_rng(seed)

    dates = pd.date_range("2019-01-01", periods=num_days, freq="B", tz="UTC")
    symbols = [f"SYN{i:03d}" for i in range(1, num_assets + 1)]

    # Markov volatility regime: calm vs stressed.
    regimes = np.zeros(num_days, dtype=int)
    transition = np.array([
        [0.985, 0.015],
        [0.070, 0.930]
    ])

    for t in range(1, num_days):
        regimes[t] = local_rng.choice([0, 1], p=transition[regimes[t - 1]])

    market_vol = np.where(regimes == 0, 0.008, 0.028)
    market_t_shocks = stats.t.rvs(df=5, size=num_days, random_state=local_rng)
    market_returns = 0.00015 + market_vol * market_t_shocks / np.sqrt(5 / (5 - 2))

    # Add rare market jumps.
    jump_mask = local_rng.uniform(size=num_days) < 0.012
    jumps = local_rng.normal(loc=-0.025, scale=0.025, size=num_days) * jump_mask
    market_returns = market_returns + jumps

    frames = []

    for symbol in symbols:
        beta = np.clip(local_rng.normal(1.0, 0.25), 0.3, 1.8)
        idio_vol = local_rng.uniform(0.006, 0.018)
        idio_df = local_rng.uniform(4, 12)
        idio_shocks = stats.t.rvs(df=idio_df, size=num_days, random_state=local_rng)
        idio_scaled = idio_shocks / np.sqrt(idio_df / (idio_df - 2))

        alpha = local_rng.normal(0.00003, 0.00008)
        returns = alpha + beta * market_returns + idio_vol * idio_scaled

        frame = pd.DataFrame({
            "symbol": symbol,
            "timestamp": dates,
            "log_return": returns,
            "market_regime": np.where(regimes == 0, "calm", "stress")
        })
        frames.append(frame)

    panel = pd.concat(frames, ignore_index=True)
    return panel

In [ ]:
previous_path = Path("data/processed/return_sanitization_v1/synthetic_sanitized_return_panel.csv")
loaded_panel = load_previous_sanitized_returns(previous_path)

if loaded_panel is not None:
    return_panel = loaded_panel.copy()
    data_source_label = "loaded_from_01_04_sanitized_panel"
else:
    return_panel = simulate_heavy_tailed_return_panel()
    data_source_label = "synthetic_heavy_tailed_panel"

return_panel.head(), data_source_label

## 4. Construct an analysis return series

There are two common choices:

1. **Pooled asset returns** — many observations, but cross-sectional dependence matters.
2. **Portfolio or index returns** — one time series, often closer to risk-management use.

This notebook focuses on an equal-weight daily portfolio return:

$$
r_{p,t} = \frac{1}{N_t}\sum_{i=1}^{N_t} r_{i,t}
$$

where $N_t$ is the number of available assets on day $t$.

This gives a single daily series for distribution fitting.

In [ ]:
def build_equal_weight_return_series(panel: pd.DataFrame) -> pd.DataFrame:
    # Build an equal-weight daily return series from a panel of asset log returns.
    daily = (
        panel
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["log_return"])
        .groupby("timestamp")
        .agg(
            log_return=("log_return", "mean"),
            n_assets=("symbol", "nunique")
        )
        .reset_index()
        .sort_values("timestamp")
    )

    daily["simple_return"] = np.expm1(daily["log_return"])
    daily["equity_curve"] = np.exp(daily["log_return"].cumsum())

    return daily


daily_returns = build_equal_weight_return_series(return_panel)

daily_returns.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(daily_returns["timestamp"], daily_returns["equity_curve"])
plt.title("Equal-Weight Synthetic Portfolio Equity Curve")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(daily_returns["timestamp"], daily_returns["log_return"])
plt.axhline(0, linewidth=1)
plt.title("Daily Equal-Weight Log Returns")
plt.xlabel("Date")
plt.ylabel("Log return")
plt.show()

## 5. Descriptive statistics

Before fitting distributions, we inspect basic statistics:

- mean;
- volatility;
- skewness;
- excess kurtosis;
- minimum and maximum;
- selected quantiles.

For a Normal distribution, excess kurtosis is 0.

Large positive excess kurtosis suggests heavier tails than Normal.

In [ ]:
def descriptive_return_statistics(returns: pd.Series) -> pd.Series:
    # Compute descriptive statistics for a return series.
    x = returns.replace([np.inf, -np.inf], np.nan).dropna()

    return pd.Series({
        "count": len(x),
        "mean_daily": x.mean(),
        "vol_daily": x.std(ddof=1),
        "mean_annualized_approx": 252 * x.mean(),
        "vol_annualized": np.sqrt(252) * x.std(ddof=1),
        "skewness": stats.skew(x, bias=False),
        "excess_kurtosis": stats.kurtosis(x, fisher=True, bias=False),
        "min": x.min(),
        "1%": x.quantile(0.01),
        "5%": x.quantile(0.05),
        "50%": x.quantile(0.50),
        "95%": x.quantile(0.95),
        "99%": x.quantile(0.99),
        "max": x.max()
    })


desc_stats = descriptive_return_statistics(daily_returns["log_return"])
desc_stats

## 6. Train/test split

To avoid judging models only in-sample, we split the return series into:

- training set: first 70%;
- test set: last 30%.

The distributions are fit on the training set and evaluated on both training and test data.

This is not a full time-series validation framework, but it is better than only reporting in-sample fit.

In [ ]:
returns = daily_returns["log_return"].replace([np.inf, -np.inf], np.nan).dropna().to_numpy()

split_idx = int(0.70 * len(returns))
train_returns = returns[:split_idx]
test_returns = returns[split_idx:]

len(train_returns), len(test_returns)

## 7. Fit Normal, Student-t, and Laplace distributions

We use maximum likelihood estimation through `scipy.stats`.

For each model, we store:

- fitted parameters;
- log-likelihood;
- AIC;
- BIC;
- train/test average log likelihood.

The Akaike Information Criterion is:

$$
\text{AIC} = 2k - 2\log L
$$

The Bayesian Information Criterion is:

$$
\text{BIC} = k\log n - 2\log L
$$

where:

- $k$ is the number of fitted parameters;
- $n$ is the number of observations;
- $L$ is the likelihood.

Lower AIC/BIC is better, but these criteria do not prove that a model is true.

In [ ]:
@dataclass
class FittedDistribution:
    # Container for fitted distribution results.
    name: str
    scipy_dist_name: str
    params: tuple[float, ...]
    num_params: int
    train_loglik: float
    test_loglik: float
    aic: float
    bic: float
    train_avg_loglik: float
    test_avg_loglik: float


def fit_candidate_distributions(train: np.ndarray, test: np.ndarray) -> dict[str, FittedDistribution]:
    # Fit Normal, Student-t, and Laplace distributions to training returns.
    candidates = {
        "Normal": stats.norm,
        "Student-t": stats.t,
        "Laplace": stats.laplace
    }

    fitted = {}
    n = len(train)

    for name, dist in candidates.items():
        params = dist.fit(train)
        train_logpdf = dist.logpdf(train, *params)
        test_logpdf = dist.logpdf(test, *params)

        train_loglik = float(np.sum(train_logpdf))
        test_loglik = float(np.sum(test_logpdf))
        k = len(params)

        aic = 2 * k - 2 * train_loglik
        bic = k * np.log(n) - 2 * train_loglik

        fitted[name] = FittedDistribution(
            name=name,
            scipy_dist_name=dist.name,
            params=tuple(float(p) for p in params),
            num_params=k,
            train_loglik=train_loglik,
            test_loglik=test_loglik,
            aic=float(aic),
            bic=float(bic),
            train_avg_loglik=float(train_loglik / len(train)),
            test_avg_loglik=float(test_loglik / len(test))
        )

    return fitted


fitted_models = fit_candidate_distributions(train_returns, test_returns)

fit_summary = pd.DataFrame([asdict(model) for model in fitted_models.values()])
fit_summary.sort_values("aic")

## 8. Model parameter interpretation

The fitted Student-t degrees-of-freedom parameter is especially important.

A lower value of $\nu$ means heavier tails.

For a Student-t distribution:

- variance exists only if $\nu > 2$;
- kurtosis exists only if $\nu > 4$.

If the fitted $\nu$ is low, the model is telling us that Normal tails are likely too thin.

In [ ]:
param_rows = []

for model in fitted_models.values():
    if model.name == "Normal":
        loc, scale = model.params
        param_rows.append({"model": model.name, "parameter": "mu", "value": loc})
        param_rows.append({"model": model.name, "parameter": "sigma", "value": scale})
    elif model.name == "Student-t":
        df, loc, scale = model.params
        param_rows.append({"model": model.name, "parameter": "nu_df", "value": df})
        param_rows.append({"model": model.name, "parameter": "loc", "value": loc})
        param_rows.append({"model": model.name, "parameter": "scale", "value": scale})
    elif model.name == "Laplace":
        loc, scale = model.params
        param_rows.append({"model": model.name, "parameter": "loc", "value": loc})
        param_rows.append({"model": model.name, "parameter": "b_scale", "value": scale})

parameter_table = pd.DataFrame(param_rows)
parameter_table

## 9. Histogram with fitted densities

We compare the empirical return histogram against fitted probability densities.

The central part of the distribution and the tails are both important.

A model that fits the centre well may still underestimate tail losses.

In [ ]:
def scipy_distribution(model: FittedDistribution):
    # Return scipy distribution object for a fitted model.
    mapping = {
        "Normal": stats.norm,
        "Student-t": stats.t,
        "Laplace": stats.laplace
    }
    return mapping[model.name]


x_grid = np.linspace(np.quantile(train_returns, 0.001), np.quantile(train_returns, 0.999), 1_000)

plt.figure(figsize=(10, 6))
plt.hist(train_returns, bins=80, density=True, alpha=0.45, label="Training returns")

for name, model in fitted_models.items():
    dist = scipy_distribution(model)
    pdf = dist.pdf(x_grid, *model.params)
    plt.plot(x_grid, pdf, linewidth=2, label=name)

plt.title("Empirical Return Histogram with Fitted Densities")
plt.xlabel("Daily log return")
plt.ylabel("Density")
plt.legend()
plt.show()

## 10. QQ plots

A QQ plot compares empirical quantiles with theoretical model quantiles.

If the model fits well, points should lie close to the 45-degree line.

Deviations in the tails reveal whether the model underestimates or overestimates extreme returns.

In [ ]:
def qq_data(sample: np.ndarray, model: FittedDistribution) -> pd.DataFrame:
    # Create QQ plot data for a fitted distribution.
    x = np.sort(sample)
    n = len(x)
    probs = (np.arange(1, n + 1) - 0.5) / n
    dist = scipy_distribution(model)
    theoretical = dist.ppf(probs, *model.params)

    return pd.DataFrame({
        "theoretical_quantile": theoretical,
        "empirical_quantile": x,
        "probability": probs
    })


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, model) in zip(axes, fitted_models.items()):
    qdf = qq_data(train_returns, model)
    ax.scatter(qdf["theoretical_quantile"], qdf["empirical_quantile"], s=8, alpha=0.45)

    lower = min(qdf["theoretical_quantile"].min(), qdf["empirical_quantile"].min())
    upper = max(qdf["theoretical_quantile"].max(), qdf["empirical_quantile"].max())
    ax.plot([lower, upper], [lower, upper], linestyle="--")

    ax.set_title(f"QQ plot: {name}")
    ax.set_xlabel("Theoretical quantile")
    ax.set_ylabel("Empirical quantile")

plt.tight_layout()
plt.show()

## 11. Probability integral transform diagnostics

For a fitted model with CDF $F$, the probability integral transform is:

$$
u_t = F(r_t)
$$

If the fitted distribution is correct, then $u_t$ should be approximately uniformly distributed on $[0,1]$.

This is a useful diagnostic because it tests the whole distribution, not only the mean and variance.

In [ ]:
def pit_values(sample: np.ndarray, model: FittedDistribution) -> np.ndarray:
    # Compute probability integral transform values under a fitted model.
    dist = scipy_distribution(model)
    u = dist.cdf(sample, *model.params)
    return np.clip(u, 1e-12, 1 - 1e-12)


pit_rows = []

for name, model in fitted_models.items():
    u = pit_values(test_returns, model)
    ks_stat, ks_pvalue = stats.kstest(u, "uniform")

    pit_rows.append({
        "model": name,
        "pit_mean": float(np.mean(u)),
        "pit_variance": float(np.var(u, ddof=1)),
        "uniform_ks_statistic": float(ks_stat),
        "uniform_ks_pvalue": float(ks_pvalue)
    })

pit_summary = pd.DataFrame(pit_rows)
pit_summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, model) in zip(axes, fitted_models.items()):
    u = pit_values(test_returns, model)
    ax.hist(u, bins=20, density=True, alpha=0.75)
    ax.axhline(1.0, linestyle="--")
    ax.set_title(f"PIT histogram: {name}")
    ax.set_xlabel("PIT value")
    ax.set_ylabel("Density")

plt.tight_layout()
plt.show()

## 12. Goodness-of-fit tests

Goodness-of-fit tests should be interpreted carefully.

The Kolmogorov-Smirnov test is simple, but the usual p-value is not exact when model parameters are estimated from the same data.

The Anderson-Darling test is often more sensitive to tail differences.

In this notebook, these tests are used as diagnostics, not as final truth.

In [ ]:
def goodness_of_fit_table(sample: np.ndarray, fitted_models: dict[str, FittedDistribution]) -> pd.DataFrame:
    # Compute diagnostic goodness-of-fit statistics.
    rows = []

    for name, model in fitted_models.items():
        dist = scipy_distribution(model)

        # KS test against fitted CDF. P-value is approximate because parameters were estimated.
        ks_stat, ks_pvalue = stats.kstest(sample, lambda x: dist.cdf(x, *model.params))

        row = {
            "model": name,
            "ks_statistic_approx": float(ks_stat),
            "ks_pvalue_approx": float(ks_pvalue)
        }

        if name == "Normal":
            ad_result = stats.anderson(sample, dist="norm")
            row["anderson_darling_statistic_normal_only"] = float(ad_result.statistic)
            row["ad_5pct_critical_value_normal_only"] = float(ad_result.critical_values[2])
        else:
            row["anderson_darling_statistic_normal_only"] = np.nan
            row["ad_5pct_critical_value_normal_only"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows)


gof_table = goodness_of_fit_table(train_returns, fitted_models)
gof_table

## 13. Tail exceedance probabilities

Risk management is especially sensitive to the tails.

We compare empirical and model-implied probabilities of large absolute moves:

$$
\mathbb P(|r - \mu| > k\sigma)
$$

for $k = 2,3,4,5$.

A Normal distribution assigns very small probabilities to large standardised moves. Empirical returns often exceed those probabilities.

In [ ]:
def tail_exceedance_table(sample: np.ndarray, fitted_models: dict[str, FittedDistribution]) -> pd.DataFrame:
    # Compare empirical and fitted two-sided tail exceedance probabilities.
    empirical_mu = np.mean(sample)
    empirical_sigma = np.std(sample, ddof=1)

    rows = []

    for k in [2, 3, 4, 5]:
        lower = empirical_mu - k * empirical_sigma
        upper = empirical_mu + k * empirical_sigma
        empirical_prob = np.mean((sample < lower) | (sample > upper))

        row = {
            "threshold_sigma": k,
            "lower_threshold": lower,
            "upper_threshold": upper,
            "empirical_probability": empirical_prob
        }

        for name, model in fitted_models.items():
            dist = scipy_distribution(model)
            model_prob = dist.cdf(lower, *model.params) + (1 - dist.cdf(upper, *model.params))
            row[f"{name}_probability"] = model_prob
            row[f"{name}_under_over_ratio"] = model_prob / empirical_prob if empirical_prob > 0 else np.nan

        rows.append(row)

    return pd.DataFrame(rows)


tail_table = tail_exceedance_table(train_returns, fitted_models)
tail_table

In [ ]:
plot_tail = tail_table[[
    "threshold_sigma",
    "empirical_probability",
    "Normal_probability",
    "Student-t_probability",
    "Laplace_probability"
]].copy()

plt.figure(figsize=(10, 6))

for col in ["empirical_probability", "Normal_probability", "Student-t_probability", "Laplace_probability"]:
    plt.plot(plot_tail["threshold_sigma"], plot_tail[col], marker="o", label=col)

plt.yscale("log")
plt.title("Two-Sided Tail Exceedance Probabilities")
plt.xlabel("Threshold in empirical standard deviations")
plt.ylabel("Probability, log scale")
plt.legend()
plt.show()

## 14. Value-at-Risk and Expected Shortfall

For a return distribution, left-tail Value-at-Risk at level $\alpha$ can be written as:

$$
\operatorname{VaR}_{\alpha} = -q_\alpha(r)
$$

where $q_\alpha(r)$ is the $\alpha$-quantile of returns.

Expected Shortfall is the average loss conditional on being beyond VaR:

$$
\begin{aligned}
\operatorname{ES}_{\alpha} &= -\mathbb E[r \mid r \leq q_\alpha(r)]
\end{aligned}
$$

Expected Shortfall is more sensitive to tail shape than VaR.

In [ ]:
def empirical_var_es(sample: np.ndarray, alpha: float) -> tuple[float, float]:
    # Empirical left-tail VaR and ES for returns.
    q = np.quantile(sample, alpha)
    tail = sample[sample <= q]
    var = -q
    es = -tail.mean()
    return float(var), float(es)


def model_var_es_by_simulation(
    model: FittedDistribution,
    alpha: float,
    num_simulations: int = 250_000,
    seed: int = 123
) -> tuple[float, float]:
    # Estimate model VaR and ES by simulation for consistent handling across distributions.
    local_rng = np.random.default_rng(seed)
    dist = scipy_distribution(model)
    simulated = dist.rvs(*model.params, size=num_simulations, random_state=local_rng)
    return empirical_var_es(simulated, alpha)


risk_rows = []

for alpha in [0.05, 0.01]:
    emp_var, emp_es = empirical_var_es(test_returns, alpha)

    row = {
        "alpha": alpha,
        "Empirical_VaR": emp_var,
        "Empirical_ES": emp_es
    }

    for name, model in fitted_models.items():
        model_var, model_es = model_var_es_by_simulation(model, alpha=alpha, seed=42)
        row[f"{name}_VaR"] = model_var
        row[f"{name}_ES"] = model_es

    risk_rows.append(row)

risk_table = pd.DataFrame(risk_rows)
risk_table

## 15. VaR exceedance backtest

A VaR model at level $\alpha$ predicts that approximately an $\alpha$ fraction of returns should fall below the model quantile.

If far more observations exceed VaR than expected, the model is underestimating risk.

For each fitted model, we compute:

$$
\begin{aligned}
\text{exceedance}_t &= \mathbf{1}\{r_t < q_\alpha^{\text{model}}\}
\end{aligned}
$$
on the test set.

In [ ]:
def kupiec_unconditional_coverage_test(num_exceedances: int, num_observations: int, alpha: float) -> tuple[float, float]:
    # Kupiec likelihood-ratio test for unconditional VaR coverage.
    # Returns LR statistic and p-value.
    x = num_exceedances
    n = num_observations

    if x == 0 or x == n:
        # Avoid log(0) in extreme cases.
        return np.nan, np.nan

    phat = x / n

    log_likelihood_null = (n - x) * np.log(1 - alpha) + x * np.log(alpha)
    log_likelihood_alt = (n - x) * np.log(1 - phat) + x * np.log(phat)

    lr_stat = -2 * (log_likelihood_null - log_likelihood_alt)
    p_value = 1 - stats.chi2.cdf(lr_stat, df=1)

    return float(lr_stat), float(p_value)


def var_backtest_table(test_sample: np.ndarray, fitted_models: dict[str, FittedDistribution], alpha: float) -> pd.DataFrame:
    # Compare model VaR exceedances on the test sample.
    rows = []

    for name, model in fitted_models.items():
        dist = scipy_distribution(model)
        q = dist.ppf(alpha, *model.params)
        exceedances = test_sample < q

        count = int(exceedances.sum())
        expected = alpha * len(test_sample)
        observed_rate = count / len(test_sample)

        lr_stat, p_value = kupiec_unconditional_coverage_test(
            num_exceedances=count,
            num_observations=len(test_sample),
            alpha=alpha
        )

        rows.append({
            "model": name,
            "alpha": alpha,
            "model_return_quantile": q,
            "model_VaR": -q,
            "num_test_observations": len(test_sample),
            "expected_exceedances": expected,
            "observed_exceedances": count,
            "observed_exceedance_rate": observed_rate,
            "kupiec_lr_stat": lr_stat,
            "kupiec_p_value": p_value
        })

    return pd.DataFrame(rows)


var_backtest_5 = var_backtest_table(test_returns, fitted_models, alpha=0.05)
var_backtest_1 = var_backtest_table(test_returns, fitted_models, alpha=0.01)

var_backtest = pd.concat([var_backtest_5, var_backtest_1], ignore_index=True)
var_backtest

In [ ]:
plot_backtest = var_backtest[var_backtest["alpha"] == 0.01].copy()

plt.figure(figsize=(10, 6))
plt.bar(plot_backtest["model"], plot_backtest["observed_exceedance_rate"])
plt.axhline(0.01, linestyle="--", label="Target 1% exceedance rate")
plt.title("1% VaR Exceedance Rate by Model")
plt.xlabel("Model")
plt.ylabel("Observed exceedance rate")
plt.legend()
plt.show()

## 16. Rolling-window instability

Distribution parameters are not constant through time.

We estimate rolling skewness and rolling excess kurtosis to show how the empirical distribution changes across regimes.

This matters because a single unconditional distribution fitted on the whole sample may average across calm and stressed periods.

In [ ]:
rolling_window = 126

rolling_stats = daily_returns[["timestamp", "log_return"]].copy()
rolling_stats["rolling_vol"] = rolling_stats["log_return"].rolling(rolling_window).std()
rolling_stats["rolling_skew"] = rolling_stats["log_return"].rolling(rolling_window).skew()
rolling_stats["rolling_excess_kurtosis"] = (
    rolling_stats["log_return"].rolling(rolling_window).kurt()
)

rolling_stats.tail()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(rolling_stats["timestamp"], rolling_stats["rolling_vol"])
plt.title("Rolling Daily Volatility")
plt.xlabel("Date")
plt.ylabel("Rolling volatility")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(rolling_stats["timestamp"], rolling_stats["rolling_excess_kurtosis"])
plt.axhline(0, linestyle="--")
plt.title("Rolling Excess Kurtosis")
plt.xlabel("Date")
plt.ylabel("Rolling excess kurtosis")
plt.show()

## 17. Model ranking summary

We combine several diagnostics:

1. in-sample AIC;
2. in-sample BIC;
3. out-of-sample average log likelihood;
4. PIT uniformity;
5. VaR exceedance diagnostics.

There is no guarantee that all diagnostics agree.

A good model for central density may still be poor for tail risk.

In [ ]:
ranking_table = fit_summary[[
    "name",
    "aic",
    "bic",
    "train_avg_loglik",
    "test_avg_loglik"
]].rename(columns={"name": "model"})

ranking_table = ranking_table.merge(
    pit_summary[["model", "uniform_ks_statistic", "uniform_ks_pvalue"]],
    on="model",
    how="left"
)

ranking_table = ranking_table.merge(
    var_backtest[var_backtest["alpha"] == 0.01][[
        "model",
        "observed_exceedance_rate",
        "kupiec_p_value"
    ]].rename(columns={
        "observed_exceedance_rate": "observed_1pct_var_exceedance_rate",
        "kupiec_p_value": "kupiec_1pct_p_value"
    }),
    on="model",
    how="left"
)

ranking_table["aic_rank"] = ranking_table["aic"].rank(method="min")
ranking_table["bic_rank"] = ranking_table["bic"].rank(method="min")
ranking_table["test_loglik_rank"] = ranking_table["test_avg_loglik"].rank(ascending=False, method="min")

ranking_table.sort_values("test_loglik_rank")

## 18. Saving outputs

The analyzer saves:

1. fitted parameter table;
2. model comparison table;
3. tail exceedance table;
4. VaR/ES table;
5. VaR backtest table;
6. descriptive statistics;
7. compact JSON report.

These outputs can be used by later notebooks on VaR, CVaR, stress testing, and portfolio construction.

In [ ]:
output_dir = Path("data/processed/empirical_distribution_analyzer_v1")
output_dir.mkdir(parents=True, exist_ok=True)

fit_summary_path = output_dir / "distribution_fit_summary.csv"
parameter_table_path = output_dir / "distribution_parameters.csv"
tail_table_path = output_dir / "tail_exceedance_table.csv"
risk_table_path = output_dir / "var_es_table.csv"
var_backtest_path = output_dir / "var_backtest_table.csv"
ranking_table_path = output_dir / "distribution_ranking_table.csv"
report_path = output_dir / "distribution_analyzer_report.json"

fit_summary.to_csv(fit_summary_path, index=False)
parameter_table.to_csv(parameter_table_path, index=False)
tail_table.to_csv(tail_table_path, index=False)
risk_table.to_csv(risk_table_path, index=False)
var_backtest.to_csv(var_backtest_path, index=False)
ranking_table.to_csv(ranking_table_path, index=False)

best_by_aic = ranking_table.sort_values("aic").iloc[0]["model"]
best_by_test_loglik = ranking_table.sort_values("test_avg_loglik", ascending=False).iloc[0]["model"]

report = {
    "schema_version": "empirical_distribution_analyzer_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "data_source_label": data_source_label,
    "num_train_observations": int(len(train_returns)),
    "num_test_observations": int(len(test_returns)),
    "descriptive_statistics": {k: float(v) for k, v in desc_stats.items()},
    "best_model_by_aic": str(best_by_aic),
    "best_model_by_test_avg_loglik": str(best_by_test_loglik),
    "candidate_models": list(fitted_models.keys()),
    "output_files": {
        "fit_summary": str(fit_summary_path),
        "parameters": str(parameter_table_path),
        "tail_exceedance": str(tail_table_path),
        "var_es": str(risk_table_path),
        "var_backtest": str(var_backtest_path),
        "ranking": str(ranking_table_path)
    }
}

report_path.write_text(json.dumps(report, indent=2))

fit_summary_path, report_path

## 19. Practical checklist for empirical distribution analysis

Before using a return distribution in a risk model, check:

1. **Sample definition**  
   Are these asset returns, strategy returns, index returns, or pooled returns?

2. **Cleaning policy**  
   Were corporate actions, bad ticks, missingness, and survivorship handled?

3. **Stationarity**  
   Are returns from one stable regime or a mixture of regimes?

4. **Central fit vs tail fit**  
   Does the model fit the centre but fail in the tails?

5. **Out-of-sample likelihood**  
   Does the model generalise beyond the fitting sample?

6. **VaR exceedances**  
   Does the model underestimate tail-loss frequency?

7. **Expected Shortfall sensitivity**  
   Does the model produce materially different ES estimates?

8. **Model purpose**  
   Is the model used for simulation, risk, option pricing, portfolio optimisation, or diagnostics?

9. **Symmetry assumption**  
   Are left and right tails equally heavy?

10. **Time variation**  
   Do parameters change through regimes?

## 20. Limitations

This notebook deliberately keeps the models simple.

### 20.1 Unconditional distributions ignore volatility clustering

Normal, Student-t, and Laplace models are fitted to the whole training sample.

They do not condition on recent volatility, market regime, or macro state.

A GARCH model with Student-t innovations may fit volatility clustering better than a static Student-t distribution.

### 20.2 Symmetry is restrictive

All three fitted distributions are symmetric.

Real return distributions may be skewed, especially for:

- equity indices;
- options strategies;
- credit portfolios;
- leveraged strategies;
- carry trades.

### 20.3 The Student-t distribution is not always enough

A Student-t model may improve tail fit compared with Normal, but it can still miss:

- skewness;
- volatility regimes;
- jumps;
- tail asymmetry;
- clustered extremes.

### 20.4 Goodness-of-fit p-values are not final truth

Tests like Kolmogorov-Smirnov and Anderson-Darling have assumptions and limitations.

When parameters are estimated from the same data, standard p-values may not be exact.

The tests are useful diagnostics, not final certification.

### 20.5 Equal-weight returns hide cross-sectional structure

The equal-weight portfolio return is easier to analyse, but it compresses information.

Individual assets may have very different tail behaviour.

### 20.6 Tail estimation is data-hungry

Extreme losses are rare by definition.

Estimating 1% or 0.1% tails from a short sample is uncertain.

This motivates stress testing, extreme value theory, scenario analysis, and Bayesian/robust approaches.

## 21. Research frontier and current directions

Empirical return modelling remains active because financial distributions are non-Gaussian, time-varying, and difficult to estimate in the tails.

### 21.1 Extreme Value Theory

Extreme Value Theory focuses directly on the tails rather than fitting the whole distribution.

A common approach is peaks-over-threshold modelling using the Generalized Pareto Distribution.

A future notebook could compare:

- Normal VaR;
- Student-t VaR;
- empirical historical VaR;
- EVT-based VaR.

### 21.2 Skewed and generalized heavy-tailed distributions

Real returns can be asymmetric.

Extensions include:

- skewed Student-t;
- generalized hyperbolic distributions;
- normal-inverse-Gaussian distributions;
- stable distributions;
- mixture models.

These can capture tail asymmetry and sharper central peaks.

### 21.3 Conditional distribution models

Rather than fitting one unconditional distribution, conditional models forecast the next-period distribution:

$$
r_{t+1} \mid \mathcal F_t
$$
Examples include:

- GARCH with Student-t innovations;
- stochastic volatility models;
- hidden Markov models;
- regime-switching distributions;
- volatility-surface-implied distributions.

### 21.4 Distributional machine learning

Modern models can forecast the full predictive distribution rather than only the mean.

Examples include:

- neural networks trained with negative log-likelihood;
- quantile regression networks;
- normalizing flows;
- diffusion models;
- conformal prediction intervals.

The challenge is to improve calibration without overfitting noisy financial data.

### 21.5 Proper scoring rules

A distributional forecast should be judged using probabilistic scores, not only point errors.

Useful tools include:

- log predictive score;
- continuous ranked probability score;
- probability integral transform;
- calibration plots;
- tail-weighted scoring rules.

### 21.6 Stress testing and model risk

A fitted distribution should not replace stress testing.

A robust risk workflow combines:

- statistical models;
- historical crises;
- hypothetical scenarios;
- liquidity stress;
- margin stress;
- model uncertainty.

A future notebook could treat Normal, Student-t, Laplace, and EVT VaR as competing model-risk scenarios.

## 22. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_01_garch_volatility_modeling**  
   Moving from unconditional distributions to conditional volatility models.

2. **03_02_har_realized_volatility_forecasting**  
   Using realised volatility to model time-varying risk.

3. **04_06_var_cvar_and_stress_testing**  
   Turning fitted distributions into risk measures and stress tests.

4. **04_11_cvar_convex_optimization**  
   Using tail-risk estimates in portfolio construction.

5. **05_06_performance_haircut_and_deflated_sharpe**  
   Understanding how non-normality affects performance metrics.

6. **05_07_purged_kfold_and_embargo_cv**  
   Validating distributional models without leakage.

7. **07_01_multi_asset_cta_research_pipeline**  
   Using distribution diagnostics as a pre-backtest quality control stage.

## 23. Summary

This notebook compared Normal, Student-t, and Laplace models for empirical return data.

The main workflow was:

1. build an equal-weight daily return series;
2. compute descriptive statistics;
3. fit candidate parametric distributions;
4. compare AIC, BIC, and out-of-sample likelihood;
5. inspect histograms and QQ plots;
6. run probability integral transform diagnostics;
7. compare tail exceedance probabilities;
8. estimate VaR and Expected Shortfall;
9. backtest VaR exceedances;
10. save model-comparison outputs.

The key computational takeaway is:

> Distribution fitting should be treated as an auditable model-selection problem, not as a one-line call to `stats.norm.fit`.

The key financial takeaway is:

> Normality is a convenient benchmark, not a default truth. Tail assumptions directly affect risk, leverage, stress tests, and portfolio decisions.

## 24. Further reading

### Distribution fitting and diagnostics

- SciPy `stats` documentation.
- NIST Engineering Statistics Handbook sections on goodness-of-fit tests.
- Anderson-Darling and Kolmogorov-Smirnov test references.
- Probability integral transform diagnostics for distributional forecasts.

### Financial return distributions

- Mandelbrot, B. "The Variation of Certain Speculative Prices."
- Fama, E. "The Behavior of Stock-Market Prices."
- Cont, R. "Empirical properties of asset returns: stylized facts and statistical issues."
- Platen, E. and Rendek, R. "Empirical Evidence on Student-t Log-Returns of Diversified World Stock Indices."
- Toth, D. and Jones, B. "Against the Norm: Modeling Daily Stock Returns with the Laplace Distribution."

### Risk and extensions

- McNeil, Frey, and Embrechts. *Quantitative Risk Management.*
- Glasserman, P. *Monte Carlo Methods in Financial Engineering.*
- Extreme Value Theory for financial risk.
- GARCH and stochastic volatility models with heavy-tailed innovations.